In [73]:
%pip install elasticsearch

python(81630) MallocStackLogging: can't turn off malloc stack logging because it was not enabled.


You should consider upgrading via the '/Users/akshaypatade/Desktop/Projects/purchase-orders/venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [1]:
from elasticsearch import Elasticsearch, exceptions
from urllib.request import urlopen
import json
import time

In [2]:
client = Elasticsearch(hosts = "https://5856681e52d64c35a4b39da6e44c832b.us-central1.gcp.cloud.es.io:443", api_key = "Rzd0UGU1TUIwRFpHXzJVU2dYaHc6STE0c1NTUUpUbUdLYk1mMUJIb21yQQ==")

In [3]:
print(client.info())

{'name': 'instance-0000000000', 'cluster_name': '5856681e52d64c35a4b39da6e44c832b', 'cluster_uuid': 'rgYgR2nNRx6vydAPWjUj_w', 'version': {'number': '8.16.1', 'build_flavor': 'default', 'build_type': 'docker', 'build_hash': 'ffe992aa682c1968b5df375b5095b3a21f122bf3', 'build_date': '2024-11-19T16:00:31.793213192Z', 'build_snapshot': False, 'lucene_version': '9.12.0', 'minimum_wire_compatibility_version': '7.17.0', 'minimum_index_compatibility_version': '7.0.0'}, 'tagline': 'You Know, for Search'}


In [52]:
try:
    client.inference.delete(inference_id="my-elser-endpoint")
except exceptions.NotFoundError:
    # Inference endpoint does not exist
    pass

try:
    client.options(
        request_timeout=60, max_retries=3, retry_on_timeout=True
    ).inference.put(
        task_type="sparse_embedding",
        inference_id="my-elser-endpoint",
        body={
            "service": "elser",
            "service_settings": {"num_allocations": 1, "num_threads": 1},
        },
    )
    print("Inference endpoint created successfully")
except exceptions.BadRequestError as e:
    if e.error == "resource_already_exists_exception":
        print("Inference endpoint created successfully")
    else:
        raise e

/var/folders/m8/9fkcwstj36jc40jlydh29f9r0000gn/T/ipykernel_14433/1915381603.py:2: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  client.inference.delete(inference_id="my-elser-endpoint")
/var/folders/m8/9fkcwstj36jc40jlydh29f9r0000gn/T/ipykernel_14433/1915381603.py:8: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  client.options(


Inference endpoint created successfully


/var/folders/m8/9fkcwstj36jc40jlydh29f9r0000gn/T/ipykernel_14433/1915381603.py:8: ElasticsearchWarning: Putting elasticsearch service inference endpoints (including elser service) without a model_id field is deprecated and will be removed in a future release. Please specify a model_id field.
  client.options(
/var/folders/m8/9fkcwstj36jc40jlydh29f9r0000gn/T/ipykernel_14433/1915381603.py:8: ElasticsearchWarning: The [elser] service is deprecated and will be removed in a future release. Use the [elasticsearch] service instead, with [model_id] set to [.elser_model_2_linux-x86_64] in the [service_settings]
  client.options(


In [53]:
inference_endpoint_info = client.inference.get(inference_id="my-elser-endpoint")
model_id = inference_endpoint_info["endpoints"][0]["service_settings"]["model_id"]

while True:
    status = client.ml.get_trained_models_stats(
        model_id=model_id,
    )

    deployment_stats = status["trained_model_stats"][0].get("deployment_stats")
    if deployment_stats is None:
        print("ELSER Model is currently being deployed.")
        time.sleep(5)
        continue

    nodes = deployment_stats.get("nodes")
    if nodes is not None and len(nodes) > 0:
        print("ELSER Model has been successfully deployed.")
        break
    else:
        print("ELSER Model is currently being deployed.")
    time.sleep(5)

ELSER Model has been successfully deployed.


/var/folders/m8/9fkcwstj36jc40jlydh29f9r0000gn/T/ipykernel_14433/734450827.py:1: GeneralAvailabilityWarning: This API is in technical preview and may be changed or removed in a future release. Elastic will work to fix any issues, but features in technical preview are not subject to the support SLA of official GA features.
  inference_endpoint_info = client.inference.get(inference_id="my-elser-endpoint")


In [61]:
client.indices.delete(index="semantic-product-search", ignore_unavailable=True)
client.indices.create(
    index="semantic-product-search",
    mappings={
        "properties": {
            "id": {"type": "text"},
            "type": {"type": "text"},
            "material" :{"type": "text"},
            "size" : {"type": "text"},
            "length" : {"type": "text"},
            "coating": {"type": "text"},
            "thread_type" : {"type": "text"},
            "description" : {"type": "text", "copy_to" : "product_search_semantic"},

            "product_search_semantic": {
                "type": "semantic_text",
                "inference_id": "my-elser-endpoint",
            },
        }
    },
)

ObjectApiResponse({'acknowledged': True, 'shards_acknowledged': True, 'index': 'semantic-product-search'})

In [7]:
def pretty_search_response(response):
    if len(response["hits"]["hits"]) == 0:
        print("Your search returned no results.")
    else:
        for hit in response["hits"]["hits"]:
            id = hit["_id"]
            score = hit["_score"]
            description = hit["_source"]["description"]

            pretty_output = f"\nID: {id}\nScore: {score} \ndescription: {description} "

            print(pretty_output)

In [8]:
#Once the data is populated in the elastic search, it is time to run the query and retrieve the elements
response = client.search(
    index="semantic-product-search",
    query={"semantic": {"field": "product_search_semantic", "query": "Steel Bolt 40mm"}},
)

pretty_search_response(response)


ID: 9c550950-5f75-4f68-9a6b-19c3270e9183
Score: 26.17872 
description: Steel Bolt 1/2" 40mm Galvanized Wood 

ID: 36a7f976-87ba-4adf-bb6e-e5635b35c329
Score: 25.938541 
description: Steel Bolt 1/4" 40mm Galvanized Fine 

ID: 34d27063-9028-4302-84a3-2e74ec52c2ab
Score: 25.924704 
description: Steel Bolt 1/2" 40mm Uncoated Fine 

ID: 87eb8643-987f-4d99-803c-8efbe62b72c3
Score: 25.787094 
description: Steel Bolt 1/4" 40mm Galvanized Wood 

ID: 34cf67e7-098f-410d-9d18-24e2328b5bab
Score: 25.47295 
description: Steel Bolt 1/2" 40mm Galvanized Fine 

ID: c5d0d2b7-3b2d-482a-9c99-b9281431b86a
Score: 25.258295 
description: Steel Bolt 1/2" 40mm Galvanized Coarse 

ID: 1d2d3eca-6c17-4ba7-baae-e4e4fd7e465b
Score: 25.164213 
description: Steel Bolt 3/4" 40mm Nickel Plated Wood 

ID: 841fedbf-30d2-4197-8af6-739b922207c7
Score: 24.999783 
description: Steel Bolt 1/2" 40mm Nickel Plated Wood 

ID: 9df43e5a-ce6b-4c51-81fe-5fc73e2d4475
Score: 24.994846 
description: Steel Bolt 1/4" 40mm Uncoated Wood 